<a href="https://colab.research.google.com/github/fmunozm-lgtm/Analisis_Datos-II/blob/main/00_verificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Verificación del entorno de trabajo

**EC2053C · Análisis de Datos II · Ayudantía 01 · Semana 1**
Universidad Católica de la Santísima Concepción · FACEA · Ingeniería en Información y Control de Gestión

---

Este cuaderno comprueba que el entorno `ec2053c` está correctamente instalado y que su computador puede
reproducir un resultado. No calcula nada del proyecto: sólo verifica que la infraestructura funciona antes de
que empiece a importar.

**Cómo se usa.** Ejecute todas las celdas en orden (`Kernel → Restart & Run All`). Si alguna falla, el propio
cuaderno le dice qué falta. Al final se escribe un informe en `reports/entorno_verificado.txt`.

**Qué se entrega.** Captura de este cuaderno ejecutado, con las salidas visibles, y el enlace al repositorio del
equipo, en EV@ antes del domingo 9 de agosto a las 23:59.

## 1. Identidad del entorno

Qué intérprete está ejecutando este cuaderno. Si la ruta no apunta al entorno `ec2053c`, el kernel seleccionado no es el correcto: cámbielo antes de seguir.

In [7]:
import sys, platform, os
from pathlib import Path

print("Python      :", sys.version.split()[0])
print("Ejecutable  :", sys.executable)
print("Sistema     :", platform.system(), platform.release())
print("Arquitectura:", platform.machine())
print("Directorio  :", Path.cwd())

entorno = Path(sys.executable).parent.parent.name
print("\nEntorno detectado:", entorno)
if "ec2053c" not in entorno.lower():
    print("AVISO: el kernel no parece ser el entorno ec2053c. Reviselo antes de continuar.")
else:
    print("OK: kernel correcto.")

Python      : 3.12.13
Ejecutable  : /usr/bin/python3
Sistema     : Linux 6.6.122+
Arquitectura: x86_64
Directorio  : /content

Entorno detectado: usr
AVISO: el kernel no parece ser el entorno ec2053c. Reviselo antes de continuar.


## 2. Paquetes requeridos

Se verifica presencia y versión mínima. Una versión menor no siempre rompe el código, pero sí rompe la reproducibilidad entre computadores del mismo equipo, que es lo que aquí importa.

In [2]:
import importlib

REQUERIDOS = {
    "numpy":        "1.26",
    "pandas":       "2.1",
    "scipy":        "1.11",
    "sklearn":      "1.4",
    "matplotlib":   "3.8",
    "pyarrow":      "14.0",
}
OPCIONALES = {
    "statsmodels":  "0.14",
    "seaborn":      "0.13",
}

def _tupla(v):
    partes = []
    for p in str(v).split("."):
        num = "".join(ch for ch in p if ch.isdigit())
        partes.append(int(num) if num else 0)
    return tuple(partes)

def revisar(paquetes, obligatorio=True):
    filas = []
    for nombre, minimo in paquetes.items():
        try:
            mod = importlib.import_module(nombre)
            version = getattr(mod, "__version__", "desconocida")
            ok = version == "desconocida" or _tupla(version) >= _tupla(minimo)
            estado = "OK" if ok else "VERSION BAJA"
        except ImportError:
            version, estado = "-", ("FALTA" if obligatorio else "opcional, no instalado")
        filas.append((nombre, minimo, version, estado))
    return filas

filas = revisar(REQUERIDOS) + revisar(OPCIONALES, obligatorio=False)

ancho = max(len(f[0]) for f in filas) + 2
print(f"{'paquete':<{ancho}}{'mínima':<10}{'instalada':<14}estado")
print("-" * (ancho + 40))
for nombre, minimo, version, estado in filas:
    print(f"{nombre:<{ancho}}{minimo:<10}{version:<14}{estado}")

faltantes = [f[0] for f in filas[:len(REQUERIDOS)] if f[3] != "OK"]
print()
if faltantes:
    print("Instale o actualice antes de seguir:")
    print("  conda install -n ec2053c " + " ".join(faltantes).replace("sklearn", "scikit-learn"))
else:
    print("OK: todos los paquetes obligatorios están disponibles.")

paquete      mínima    instalada     estado
-----------------------------------------------------
numpy        1.26      2.0.2         OK
pandas       2.1       2.2.2         OK
scipy        1.11      1.16.3        OK
sklearn      1.4       1.6.1         OK
matplotlib   3.8       3.10.0        OK
pyarrow      14.0      18.1.0        OK
statsmodels  0.14      0.14.6        OK
seaborn      0.13      0.13.2        OK

OK: todos los paquetes obligatorios están disponibles.


## 3. Semilla y reproducibilidad

Todo el semestre usa la misma semilla. Si el número que imprime esta celda no coincide con el de su compañero de equipo, hay una diferencia de versión que va a aparecer más tarde en forma de resultados que no se pueden replicar.

In [3]:
import numpy as np
import random

SEED = 2053

random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

muestra = rng.normal(size=5)
print("Semilla del curso :", SEED)
print("Muestra normal    :", np.round(muestra, 6))
print("Suma de control   :", round(float(muestra.sum()), 10))

ESPERADO = 2.6972271071
coincide = abs(float(muestra.sum()) - ESPERADO) < 1e-9
print("\nValor de referencia:", ESPERADO)
print("OK: reproducible." if coincide else
      "AVISO: la suma no coincide con la referencia. Revise la versión de numpy con su equipo.")

Semilla del curso : 2053
Muestra normal    : [-0.13432   1.251593 -0.449899  2.36988  -0.340027]
Suma de control   : 2.6972271071

Valor de referencia: 2.6972271071
OK: reproducible.


## 4. Estructura de carpetas del proyecto

El repositorio del equipo usa siempre la misma estructura. No es una formalidad: la reproducibilidad de la entrega final depende de que cualquier persona sepa dónde está cada cosa sin preguntar.

In [4]:
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CARPETAS = [
    "data/raw",         # extractos originales, nunca se editan
    "data/interim",     # pasos intermedios
    "data/processed",   # matrices de diseño listas para modelar
    "notebooks",        # cuadernos numerados
    "src",              # funciones reutilizables
    "reports",          # informes y figuras
]

for c in CARPETAS:
    (RAIZ / c).mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ, "\n")
for c in CARPETAS:
    ruta = RAIZ / c
    n = len([p for p in ruta.iterdir() if p.is_file()]) if ruta.exists() else 0
    print(f"  {c:<18} {'existe' if ruta.exists() else 'FALTA':<8} {n} archivo(s)")

gitignore = RAIZ / ".gitignore"
if not gitignore.exists():
    gitignore.write_text(
        "data/raw/\ndata/interim/\n.ipynb_checkpoints/\n__pycache__/\n*.pyc\n.env\n",
        encoding="utf-8")
    print("\n.gitignore creado.")
else:
    print("\n.gitignore ya existe.")

print("\nRecordatorio: los datos de la organización NO se suben al repositorio. Sólo el código y la "
      "documentación.")

Raíz del proyecto: /content 

  data/raw           existe   0 archivo(s)
  data/interim       existe   0 archivo(s)
  data/processed     existe   0 archivo(s)
  notebooks          existe   0 archivo(s)
  src                existe   0 archivo(s)
  reports            existe   1 archivo(s)

.gitignore ya existe.

Recordatorio: los datos de la organización NO se suben al repositorio. Sólo el código y la documentación.


## 5. Prueba de humo de extremo a extremo

Un flujo mínimo con la misma estructura que usaremos todo el semestre: `Pipeline`, `ColumnTransformer` y validación cruzada. Si esta celda corre, el entorno sirve para el curso.

In [5]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

n = 300
X = pd.DataFrame({
    "monto":    rng.lognormal(mean=10, sigma=0.6, size=n),
    "atraso":   rng.poisson(lam=4, size=n).astype(float),
    "segmento": rng.choice(["Retail", "Mayorista", "Institucional"], size=n),
})
X.loc[rng.choice(n, size=15, replace=False), "atraso"] = np.nan   # faltantes a propósito
y = (X["atraso"].fillna(0) + rng.normal(0, 2, size=n) > 5).astype(int)

num = Pipeline([("imp", SimpleImputer(strategy="median")),
                ("esc", StandardScaler())])
cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                ("oh",  OneHotEncoder(handle_unknown="ignore"))])

pre = ColumnTransformer([("num", num, ["monto", "atraso"]),
                         ("cat", cat, ["segmento"])])

modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
auc = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")

print("AUC por pliegue :", np.round(auc, 4))
print(f"AUC media       : {auc.mean():.4f}  (±{auc.std():.4f})")
print("\nOK: el flujo completo se ejecuta." if auc.mean() > 0.5 else "AVISO: revise la instalación.")

AUC por pliegue : [0.9114 0.8357 0.9281 0.7612 0.8312]
AUC media       : 0.8535  (±0.0604)

OK: el flujo completo se ejecuta.


## 6. Informe de verificación

Se guarda un archivo de texto con el resultado. Ese archivo es el respaldo de que el entorno quedó operativo en su equipo, y sirve para diagnosticar diferencias entre integrantes.

In [6]:
from datetime import datetime

lineas = [
    "VERIFICACIÓN DE ENTORNO · EC2053C Análisis de Datos II · 2-2026",
    "=" * 66,
    f"Fecha de ejecución : {datetime.now():%Y-%m-%d %H:%M}",
    f"Python             : {sys.version.split()[0]}",
    f"Ejecutable         : {sys.executable}",
    f"Sistema            : {platform.system()} {platform.release()} ({platform.machine()})",
    f"Semilla del curso  : {SEED}",
    f"Reproducibilidad   : {'OK' if coincide else 'REVISAR'}",
    f"Prueba de humo AUC : {auc.mean():.4f}",
    "",
    "Paquetes:",
]
for nombre, minimo, version, estado in filas:
    lineas.append(f"  {nombre:<14} {version:<14} {estado}")

informe = "\n".join(lineas)
destino = RAIZ / "reports" / "entorno_verificado.txt"
destino.write_text(informe, encoding="utf-8")

print(informe)
print("\n" + "-" * 66)
print("Informe guardado en:", destino)

VERIFICACIÓN DE ENTORNO · EC2053C Análisis de Datos II · 2-2026
Fecha de ejecución : 2026-08-15 21:47
Python             : 3.12.13
Ejecutable         : /usr/bin/python3
Sistema            : Linux 6.6.122+ (x86_64)
Semilla del curso  : 2053
Reproducibilidad   : OK
Prueba de humo AUC : 0.8535

Paquetes:
  numpy          2.0.2          OK
  pandas         2.2.2          OK
  scipy          1.16.3         OK
  sklearn        1.6.1          OK
  matplotlib     3.10.0         OK
  pyarrow        18.1.0         OK
  statsmodels    0.14.6         OK
  seaborn        0.13.2         OK

------------------------------------------------------------------
Informe guardado en: /content/reports/entorno_verificado.txt


---

## Qué se entrega

| Elemento | Dónde | Plazo |
|---|---|---|
| Captura de este cuaderno ejecutado, con las salidas visibles | EV@ | domingo 9 de agosto, 23:59 |
| Enlace al repositorio del equipo en GitHub, con `README` y `.gitignore` | EV@ | domingo 9 de agosto, 23:59 |

El correo electrónico no es medio válido de entrega. Un cuaderno sin salidas no acredita nada, porque no
demuestra que el código corrió en su computador.

**Si algo falla.** Traiga el archivo `reports/entorno_verificado.txt` al horario de atención del jueves. El
diagnóstico es mucho más rápido con ese archivo a la vista que con una descripción del error de memoria.